# Baseline: TDVP (RBM) vs exact dynamics

Spin-1/2 chain with all-to-all XX couplings and a field along Z:

$$H = h \sum_i Z_i + \sum_{i<j} J_{ij} X_i X_j, \qquad N = 6,\ h = 10,\ J_{ij} = 1.$$

The initial state is a complex RBM ($\alpha = 1$) with seeded parameters (`SEED = 45`), held exactly as a `FullSumState`. Its vector is also the initial condition of the exact evolution $|\psi(t)\rangle = e^{-iHt}|\psi(0)\rangle$ (`scipy.sparse.linalg.expm_multiply`), so every difference comes from the TDVP projection and the integrator.

TDVP settings: RK45 with `dt = 1e-3` (forced onto the same time grid as ED), `QGTJacobianDense(holomorphic=True)` with `diag_shift = 1e-6`, SVD solver. With `diag_shift = 1e-4` the regularization alone gives an error of about $10^{-3}$ on $\langle Z \rangle$ at $t = 0.5$.

The run writes to `outputs/tdvp_vs_ed_baseline/<timestamp>/`: the TDVP JSON log and the arrays in `exp/`, the figure in `plots/`. The same comparison, at $T = 0.5$, is the CI test `tests/QRC/NQS/dynamics/test_tdvp_baseline.py`.

In [ ]:
from QRC.NQS.dynamics import BASELINE_SEED, TDVPSettings, tdvp_vs_exact
from QRC.NQS.paths import experiment_paths
from QRC.NQS.utils.outputs import find_repo_root

N = 6
H_FIELD = 10.0
J = 1.0
T = 5.0
DT = 1e-3
SEED = BASELINE_SEED  # 45

paths = experiment_paths(
    experiment_name="tdvp_vs_ed_baseline", outputs_root=find_repo_root() / "outputs"
)
print("run folder:", paths.root)

In [ ]:
settings = TDVPSettings(dt=DT, integrator="RK45", diag_shift=1e-6, linear_solver="svd")
result = tdvp_vs_exact(
    n_spins=N, h=H_FIELD, J=J, T=T, seed=SEED, settings=settings, paths=paths
)
print("time points (ED, TDVP):", len(result.times), len(result.tdvp_times))
print(f"max |<Z>_TDVP - <Z>_ED| = {result.max_abs_error:.2e}")
print(f"final fidelity |<psi_ED(T)|psi_TDVP(T)>|^2 = {result.final_fidelity:.6f}")
drift = abs(result.tdvp_energy - result.exact_energy[0]).max()
print(f"energy drift max |E_TDVP(t) - E(0)| = {drift:.2e}")

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(result.plot)))